# Protocol: pairs, rendering (Table 1 CNT + TM `mid`)

Samples the 1,000 stratified DTD pairs and renders Texture Mixer's native `mid` plus the four CNT operators (field-linear, primitive-linear, soft OT, OT-sym); also the per-pair observer scores behind the older 1,000-pair prose (Inception/DINO realism .963/.962, p=.28, BEiTv2 fidelity).

**Needs:** DTD, the CNT checkpoint (pinned commit), the Texture Mixer earth-texture checkpoint.

**Maintained runnable path:** `experiments/cnt_1000.py` (CNT block) and `experiments/tm_1000.py` (Texture Mixer `mid`) reproduce the same rendering; this notebook is kept for provenance -- it is the original source and, per INVENTORY.md, Table 1's CNT images are not bit-reproducible (this notebook used an unseeded `torch.randperm`; `matching_ablation.ipynb` onward fixed that with per-pair seeding).

In [ ]:
# ============================================================
# PHASE 1, CELL 1 — TensorFlow only. No torch in this phase.
# ============================================================
import os, sys, types, importlib, importlib.util, importlib.machinery, glob
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
ImageFile.LOAD_TRUNCATED_IMAGES = True

from google.colab import drive
drive.mount('/content/drive')
BASE = os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
TM_STORE = f'{BASE}/texturemixer'
TM_OUT = f'{TM_STORE}/midpoints'
os.makedirs(TM_OUT, exist_ok=True)

# --- repo ---
os.chdir('/content')
if not os.path.isdir('/content/TextureMixer'):
    !git clone -q https://github.com/ningyu1991/TextureMixer.git
if not os.path.isdir('/content/TextureMixer/tensorflow_vgg'):
    !git clone -q https://github.com/machrisaa/tensorflow-vgg.git /content/TextureMixer/tensorflow_vgg
os.chdir('/content/TextureMixer'); sys.path.insert(0, '/content/TextureMixer')

# --- Python 3.12 removed `imp` ---
if 'imp' not in sys.modules:
    _imp = types.ModuleType('imp')
    def _load_source(name, pathname, file=None):
        spec = importlib.util.spec_from_file_location(name, pathname)
        mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod
        spec.loader.exec_module(mod); return mod
    def _find_module(name, path=None):
        spec = importlib.machinery.PathFinder().find_spec(name, path)
        if spec is None: raise ImportError(name)
        return (None, spec.origin, ('.py','r',1))
    _imp.load_source=_load_source; _imp.find_module=_find_module
    _imp.new_module=lambda n: types.ModuleType(n); _imp.reload=importlib.reload
    _imp.acquire_lock=lambda: None; _imp.release_lock=lambda: None; _imp.PY_SOURCE=1
    sys.modules['imp'] = _imp
    print('imp shim installed')

# --- TF1 graph mode on TF2 ---
import tensorflow as _tf2
print('TF', _tf2.__version__, '| GPU:', _tf2.config.list_physical_devices('GPU'))
tf1 = _tf2.compat.v1
tf1.disable_eager_execution(); tf1.disable_v2_behavior()
tf1.logging.set_verbosity(tf1.logging.ERROR)
if not hasattr(tf1, 'Dimension'): tf1.Dimension = getattr(_tf2, 'Dimension', int)

!pip install -q tf_slim
import tf_slim as _slim
_contrib = types.ModuleType('tensorflow.contrib')
_contrib.slim=_slim; _contrib.framework=_slim; _contrib.layers=_slim
sys.modules['tensorflow.contrib']=_contrib
sys.modules['tensorflow.contrib.slim']=_slim
sys.modules['tensorflow.contrib.framework']=_slim
sys.modules['tensorflow.contrib.layers']=_slim
tf1.contrib=_contrib
sys.modules['tensorflow'] = tf1                 # repo does `import tensorflow as tf`
print('tf_slim wired into tf.contrib')

# --- Python-3 syntax fixes in the repo ---
_p='/content/TextureMixer/tfutil.py'; _s=open(_p).read()
_s=_s.replace('v.shape.ndims is 0','v.shape.ndims == 0').replace('v.shape.ndims is 1','v.shape.ndims == 1')
open(_p,'w').write(_s)

# --- checkpoint (cached on Drive) ---
CKPT = f'{TM_STORE}/earth_texture/network-final.pkl'
if not os.path.exists(CKPT):
    !pip install -q gdown
    !gdown -q 1ObAFBPGaRJFo11LUa0qNhRX14nTEWKC1 -O /content/earth.zip
    !cd /content && unzip -oq earth.zip -d "{TM_STORE}"
print('checkpoint present:', os.path.exists(CKPT))

# --- load networks (9-tuple, bind by name) ---
import tfutil, misc
sess = tfutil.create_session(force_as_default=True)
obj = misc.load_pkl(CKPT)
nets = {getattr(o,'name',f'_{i}'): o for i,o in enumerate(obj)}
print('components:', list(nets))
Es_zg, Es_zl, Gs = nets['Es_zg'], nets['Es_zl'], nets['Gs']
print('LOADED OK — Gs', Gs.input_shapes, '->', Gs.output_shapes)

In [ ]:
# ============================================================
# PHASE 1, CELL 2 — Texture Mixer interpolation. Verify first.
# ============================================================
import random
TM_RES = 128
LATENT_IDX = 0          # 0 = mean (verified); 1 = second moment

def tm_prep(pil):
    a = np.asarray(pil.convert('RGB').resize((TM_RES,)*2), dtype=np.float32)
    return (a/127.5 - 1.0).transpose(2,0,1)[None]

def tm_to_pil(arr):
    return Image.fromarray(np.clip((arr[0].transpose(1,2,0)+1.0)*127.5,0,255).astype(np.uint8))

def tm_encode(pil):
    x = tm_prep(pil)
    return (Es_zg.run(x, return_as_list=True)[LATENT_IDX],
            Es_zl.run(x, return_as_list=True)[LATENT_IDX])

def tm_generate(zg, zl):
    zg_t = np.tile(zg, (1,1,zl.shape[2],zl.shape[3]))
    return tm_to_pil(Gs.run(zg_t, zl, return_as_list=True)[0])

def texturemixer_interp(A, B, weight=0.5):
    zgA, zlA = tm_encode(A); zgB, zlB = tm_encode(B)
    return tm_generate((1-weight)*zgA + weight*zgB, (1-weight)*zlA + weight*zlB)

# --- DTD (a few reads, with retry for Drive EIO) ---
DTD_ROOT = f'{BASE}/dtd/images'
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
def safe_open(p, tries=4):
    import time as _t
    for i in range(tries):
        try:
            im = Image.open(p); im.load(); return im.convert('RGB')
        except OSError:
            if i == tries-1: raise
            _t.sleep(0.5*(i+1))
print(f'{len(cats)} categories')

# --- reconstruction sanity check ---
probe = safe_open(dtd_path(cats[20], 7))
zg, zl = tm_encode(probe)
fig, ax = plt.subplots(1,2, figsize=(6,3.2))
ax[0].imshow(probe.resize((TM_RES,)*2)); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(tm_generate(zg,zl)); ax[1].set_title('Gs(E(input))'); ax[1].axis('off')
plt.tight_layout(); plt.show()
print('>>> if the right panel resembles the left, continue to cell 3')

In [ ]:
# ============================================================
# PHASE 1, CELL 3 — TM midpoints at scale: N_PAIRS=1000.
# Stratified pair sampler (seeded, cross-category, no repeats), chunked +
# RESUMABLE (safe to reconnect mid-run), manifest to Drive.
# Swap control: full 1000 (it's only ~6 extra minutes at 0.375 s/mid).
# Expected wall time: ~15-20 min total for A/B/mid/mid_swap.
# ============================================================
import os, json, time, random, itertools
import numpy as np
from PIL import Image

N_PAIRS   = 1000
SEED      = 2026
BASE      = os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
OUT_ROOT  = f'{BASE}/tm_scale1000'
DTD_ROOT  = f'{BASE}/dtd/images'
for d in ['A', 'B', 'mid', 'mid_swap']:
    os.makedirs(f'{OUT_ROOT}/{d}', exist_ok=True)

# --- stratified sampler: uniform over unordered category pairs, then a
# fresh file from each side; seeded so Phase 2 rebuilds identical pairs
cats = sorted(c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}'))
rng = random.Random(SEED)
cat_pairs = list(itertools.combinations(cats, 2))
manifest = []
for k in range(N_PAIRS):
    cA, cB = rng.choice(cat_pairs)
    fA = rng.choice(sorted(os.listdir(f'{DTD_ROOT}/{cA}')))
    fB = rng.choice(sorted(os.listdir(f'{DTD_ROOT}/{cB}')))
    manifest.append(dict(k=k, cA=cA, fA=fA, cB=cB, fB=fB))
with open(f'{OUT_ROOT}/manifest.json', 'w') as f:
    json.dump(dict(seed=SEED, n=N_PAIRS, pairs=manifest), f)
print(f'{N_PAIRS} pairs sampled over {len(cat_pairs)} category pairs; manifest saved')

def load128(cat, fn):
    return Image.open(f'{DTD_ROOT}/{cat}/{fn}').convert('RGB').resize((128, 128))

times, done = [], 0
for m in manifest:
    k = m['k']
    outs = {d: f'{OUT_ROOT}/{d}/{k:04d}.png' for d in ['A', 'B', 'mid', 'mid_swap']}
    if all(os.path.exists(p) for p in outs.values()):
        done += 1; continue                      # resume support
    A, B = load128(m['cA'], m['fA']), load128(m['cB'], m['fB'])
    A.save(outs['A']); B.save(outs['B'])
    t0 = time.time()
    texturemixer_interp(A, B, 0.5).save(outs['mid'])
    times.append(time.time() - t0)
    texturemixer_interp(B, A, 0.5).save(outs['mid_swap'])
    if k % 100 == 0:
        print(f'  {k}/{N_PAIRS}  (median {np.median(times):.3f}s/mid)' if times
              else f'  {k}/{N_PAIRS}')
print(f'done. {done} pre-existing, {len(times)} generated, '
      f'median TM time/midpoint: {np.median(times):.3f}s')
print('>>> PHASE 1 COMPLETE. Runtime -> Restart session. Then run cells 4+.')

In [ ]:
# ============================================================
# PHASE 2, CELL 4 — torch/CNT only. Run AFTER restarting.
# ============================================================
!pip install -q pot plenoptic einops timm
import os, sys, glob, random, time, math
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon
from scipy.linalg import sqrtm
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE = os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
TM_OUT = f'{BASE}/texturemixer/midpoints'
OUT = f'{BASE}/tm_comparison'; os.makedirs(OUT, exist_ok=True)
TMP = '/content/tmp_imgs'; os.makedirs(TMP, exist_ok=True)

# --- DTD on local disk (Drive FUSE throws EIO on bursty reads) ---
LOCAL_DTD = '/content/dtd_images'
if len(glob.glob(f'{LOCAL_DTD}/*/*.jpg')) < 5000:
    print('staging DTD locally (a few minutes, once per session)...')
    os.system(f'cp -r "{BASE}/dtd/images" {LOCAL_DTD}')
DTD_ROOT = LOCAL_DTD
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
print(f'{len(cats)} categories, {len(glob.glob(f"{DTD_ROOT}/*/*.jpg"))} images')

# --- CNT ---
os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT_DIR='/content/cnt-siga24'; os.chdir(CNT_DIR); sys.path.insert(0,CNT_DIR)
from src.inference.core import TexAutoEncoderInference, load_image
model = TexAutoEncoderInference(ckpt_path=Path(f"{CNT_DIR}/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
FMS = [round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]

_tc=[0]
def cnt_load(x, size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x), size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp)
    return load_image(Path(fp), size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob, L["layer_level_layouts"][-1][0]["blob_features"].clone(), \
                     L["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(blob_ref, bf, si):
    with torch.no_grad():
        L=model.splat(blob_ref)
        L["layer_level_layouts"][-1][0]["blob_features"]=bf
        L["layer_level_layouts"][-1][0]["score_img"]=si
        return model.decode(L)
def build_style_maps(bf,si,fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref, sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def sinkhorn_transport(src,tgt,eps=0.05,n_iters=100):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=src.device)/N; b=torch.ones(M,1,device=src.device)/M; u=torch.ones(N,1,device=src.device)
    for _ in range(n_iters): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    P=u*K*v.T; return (P/(P.sum(1,keepdim=True)+1e-10))@tgt
def build_d(ia, ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def interp_ot(d, eta, n_sub=4096):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); src,tgt=sa[L][0],sb[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    i_s=torch.randperm(N,device=device)[:n_sub]; i_t=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    it=((1-eta)*sf+eta*tt[nn]).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_a"], {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_ot_sym(d, eta, n_sub=4096):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; C,H,W=A.shape
    af=A.permute(1,2,0).reshape(-1,C); bf=B.permute(1,2,0).reshape(-1,C); N=af.shape[0]
    ia=torch.randperm(N,device=device)[:n_sub]; ib=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad():
        tab=sinkhorn_transport(af[ia],bf[ib]); tba=sinkhorn_transport(bf[ib],af[ia])
    na=torch.cdist(af,af[ia]).argmin(1); nb=torch.cdist(bf,bf[ib]).argmin(1)
    it=(0.5*(((1-eta)*af+eta*tab[na])+(eta*bf+(1-eta)*tba[nb]))).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_a"], {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_lin_pixel(d, eta):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"], {s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def interp_lin_texton(d, eta):
    return decode_interp(d["blob_a"], (1-eta)*d["bf_a"]+eta*d["bf_b"], (1-eta)*d["si_a"]+eta*d["si_b"])
FNS = {'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton,
       'ot':interp_ot,'ot_sym':interp_ot_sym}

# --- observers ---
import timm
dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).to(device).eval()
for p in dino.parameters(): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
_S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
@torch.no_grad()
def dino_feats(pils, bs=16):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=518,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(dino(b).cpu().numpy())
    return np.concatenate(out)
def dino_dist(p1,p2):
    f=dino_feats([p1,p2]); return float(np.linalg.norm(f[0]-f[1]))
from torchvision.models import inception_v3, Inception_V3_Weights
_inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True).to(device).eval()
_inc.fc = torch.nn.Identity()
for p in _inc.parameters(): p.requires_grad_(False)
@torch.no_grad()
def inc_feats(pils, bs=24):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=299,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(_inc(b).cpu().numpy())
    return np.concatenate(out)
import plenoptic as po
ps_model = po.models.PortillaSimoncelli((IMG_SIZE,IMG_SIZE), n_scales=4, n_orientations=4, spatial_corr_width=7).to(device)
def _raw_ps(pil):
    g=pil.convert('L').resize((IMG_SIZE,)*2)
    x=torch.from_numpy(np.asarray(g).copy()).float()[None,None].to(device)/255.
    with torch.no_grad(): return ps_model(x).detach().cpu().numpy().ravel()
VALID = ~np.isnan(_raw_ps(Image.effect_noise((256,256),60)))
def ps_stats(pil): return _raw_ps(pil)[VALID]

vortex_feats = None
try:
    os.system('git clone -q https://github.com/scabini/VORTEX.git /content/VORTEX 2>/dev/null')
    _cwd=os.getcwd(); os.chdir('/content/VORTEX'); sys.path.insert(0,'/content/VORTEX')
    from models import VORTEX
    from torchvision import transforms
    vortex_model = VORTEX('beitv2_base_patch16_224.in1k_ft_in22k_in1k', 224).to(device).eval()
    for p in vortex_model.parameters(): p.requires_grad_(False)
    vtf = transforms.Compose([transforms.ToTensor(),
          transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)), transforms.Resize((224,224))])
    def vortex_feats(pil):
        with torch.no_grad(): return vortex_model(vtf(pil.convert('RGB')).unsqueeze(0).to(device)).cpu().numpy().ravel()
    os.chdir(_cwd); os.chdir(CNT_DIR)
    print('VORTEX loaded')
except Exception as e:
    os.chdir(CNT_DIR); print('VORTEX unavailable, continuing with 3 observers:', type(e).__name__)

print('\nPHASE 2 SETUP READY. observers:',
      ['inception','dinov2','portilla-simoncelli'] + (['vortex'] if vortex_feats else []))

In [ ]:
# ============================================================
# POST-RESTART: torch/CNT side. Do NOT run any TF cell after this.
# ============================================================
!pip install -q pot plenoptic einops timm
import os, sys, glob, random, time, gc, math
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon
from scipy.linalg import sqrtm
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'

BASE = os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
TM_OUT = f'{BASE}/texturemixer/midpoints'
OUT = f'{BASE}/baselines_texturemixer'; os.makedirs(OUT, exist_ok=True)
TMP = '/content/tmp_imgs'; os.makedirs(TMP, exist_ok=True)

# --- DTD on local disk (Drive FUSE throws EIO under bursty reads) ---
LOCAL_DTD = '/content/dtd_images'
if len(glob.glob(f'{LOCAL_DTD}/*/*.jpg')) < 5000:
    os.system(f'cp -r "{BASE}/dtd/images" {LOCAL_DTD}')
DTD_ROOT = LOCAL_DTD
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
print(f'{len(cats)} categories, {len(glob.glob(f"{DTD_ROOT}/*/*.jpg"))} images')

# --- CNT ---
os.chdir('/content'); os.system('rm -rf cnt-siga24')
os.system('git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system('pip install -q -r cnt-siga24/requirements.in')
os.system('pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT_DIR='/content/cnt-siga24'; os.chdir(CNT_DIR); sys.path.insert(0,CNT_DIR)
from src.inference.core import TexAutoEncoderInference, load_image
model = TexAutoEncoderInference(ckpt_path=Path(f"{CNT_DIR}/ckpts/finetuned-model256.ckpt"), device=device)
IMG_SIZE = 256
FMS = [round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]

_tc=[0]
def cnt_load(x, size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x), size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp)
    return load_image(Path(fp), size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy()
    return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob, L["layer_level_layouts"][-1][0]["blob_features"].clone(), \
                     L["layer_level_layouts"][-1][0]["score_img"].clone()
def decode_interp(blob_ref, bf, si):
    with torch.no_grad():
        L=model.splat(blob_ref)
        L["layer_level_layouts"][-1][0]["blob_features"]=bf
        L["layer_level_layouts"][-1][0]["score_img"]=si
        return model.decode(L)
def build_style_maps(bf,si,fms):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(fms):
        s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref, sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]
        sp={}; s=si; sp[s.shape[-1]]=s
        while s.shape[-1]>min(fms):
            s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1)
        x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def sinkhorn_transport(src,tgt,eps=0.05,n_iters=100):
    N,M=src.shape[0],tgt.shape[0]; C=torch.cdist(src,tgt).pow(2); K=torch.exp(-C/eps)
    a=torch.ones(N,1,device=src.device)/N; b=torch.ones(M,1,device=src.device)/M; u=torch.ones(N,1,device=src.device)
    for _ in range(n_iters): v=b/(K.T@u+1e-10); u=a/(K@v+1e-10)
    P=u*K*v.T; return (P/(P.sum(1,keepdim=True)+1e-10))@tgt
def build_d(ia, ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def interp_ot(d, eta, n_sub=4096):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); src,tgt=sa[L][0],sb[L][0]; C,H,W=src.shape
    sf=src.permute(1,2,0).reshape(-1,C); tf=tgt.permute(1,2,0).reshape(-1,C); N=sf.shape[0]
    i_s=torch.randperm(N,device=device)[:n_sub]; i_t=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad(): tt=sinkhorn_transport(sf[i_s],tf[i_t])
    nn=torch.cdist(sf,sf[i_s]).argmin(1)
    it=((1-eta)*sf+eta*tt[nn]).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_a"], {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_ot_sym(d, eta, n_sub=4096):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; C,H,W=A.shape
    af=A.permute(1,2,0).reshape(-1,C); bf=B.permute(1,2,0).reshape(-1,C); N=af.shape[0]
    ia=torch.randperm(N,device=device)[:n_sub]; ib=torch.randperm(N,device=device)[:n_sub]
    with torch.no_grad():
        tab=sinkhorn_transport(af[ia],bf[ib]); tba=sinkhorn_transport(bf[ib],af[ia])
    na=torch.cdist(af,af[ia]).argmin(1); nb=torch.cdist(bf,bf[ib]).argmin(1)
    it=(0.5*(((1-eta)*af+eta*tab[na])+(eta*bf+(1-eta)*tba[nb]))).reshape(H,W,C).permute(2,0,1).unsqueeze(0)
    return decode_with_style_maps(d["blob_a"], {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)})
def interp_lin_pixel(d, eta):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"], {s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def interp_lin_texton(d, eta):
    return decode_interp(d["blob_a"], (1-eta)*d["bf_a"]+eta*d["bf_b"], (1-eta)*d["si_a"]+eta*d["si_b"])
FNS = {'pixel_linear':interp_lin_pixel,'texton_linear':interp_lin_texton,
       'ot':interp_ot,'ot_sym':interp_ot_sym}

# --- observers ---
import timm
dino = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True, num_classes=0).to(device).eval()
for p in dino.parameters(): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1)
_S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
@torch.no_grad()
def dino_feats(pils, bs=16):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=518,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(dino(b).cpu().numpy())
    return np.concatenate(out)
def dino_dist(p1,p2):
    f=dino_feats([p1,p2]); return float(np.linalg.norm(f[0]-f[1]))
from torchvision.models import inception_v3, Inception_V3_Weights
_inc = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True).to(device).eval()
_inc.fc = torch.nn.Identity()
for p in _inc.parameters(): p.requires_grad_(False)
@torch.no_grad()
def inc_feats(pils, bs=24):
    out=[]
    for i in range(0,len(pils),bs):
        xs=[torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils[i:i+bs]]
        b=torch.stack(xs).to(device); b=F.interpolate(b,size=299,mode='bilinear',align_corners=False); b=(b-_M)/_S
        out.append(_inc(b).cpu().numpy())
    return np.concatenate(out)
import plenoptic as po
ps_model = po.models.PortillaSimoncelli((IMG_SIZE,IMG_SIZE), n_scales=4, n_orientations=4, spatial_corr_width=7).to(device)
def _raw_ps(pil):
    g=pil.convert('L').resize((IMG_SIZE,)*2)
    x=torch.from_numpy(np.asarray(g).copy()).float()[None,None].to(device)/255.
    with torch.no_grad(): return ps_model(x).detach().cpu().numpy().ravel()
VALID = ~np.isnan(_raw_ps(Image.effect_noise((256,256),60)))
def ps_stats(pil): return _raw_ps(pil)[VALID]
os.system('git clone -q https://github.com/scabini/VORTEX.git /content/VORTEX 2>/dev/null')
for m in [m for m in list(sys.modules) if m=='models' or m.startswith('models.')]: del sys.modules[m]
cwd=os.getcwd(); os.chdir('/content/VORTEX'); sys.path.insert(0,'/content/VORTEX')
from models import VORTEX
from torchvision import transforms
vortex_model = VORTEX('beitv2_base_patch16_224.in1k_ft_in22k_in1k', 224).to(device).eval()
for p in vortex_model.parameters(): p.requires_grad_(False)
vtf = transforms.Compose([transforms.ToTensor(),
      transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)), transforms.Resize((224,224))])
def vortex_feats(pil):
    with torch.no_grad(): return vortex_model(vtf(pil.convert('RGB')).unsqueeze(0).to(device)).cpu().numpy().ravel()
os.chdir(CNT_DIR)
print('\nCNT + observers ready')

In [ ]:
# ============================================================
# fidelity_vx observer — raw BEiTv2 via timm, replacing VORTEX.
# Tries current and legacy model names; verifies with a forward pass
# BEFORE declaring itself ready. Defines vx_extract(pil) -> (D,) tensor.
# ============================================================
import timm, torch
from torchvision import transforms

CANDIDATES = [
    'beitv2_base_patch16_224.in1k_ft_in22k_in1k',
    'beitv2_base_patch16_224.in1k_ft_in1k',
    'beitv2_base_patch16_224',
]
vx_model = None
for name in CANDIDATES:
    try:
        m = timm.create_model(name, pretrained=True, num_classes=0)
        if m is None: raise RuntimeError('factory returned None')
        vx_model = m.to(device).eval(); VX_NAME = name
        break
    except Exception as e:
        print(f'  {name}: {type(e).__name__}: {e}')
if vx_model is None:
    hits = [n for n in timm.list_models('beitv2*', pretrained=True)]
    raise RuntimeError(f'no candidate worked; available beitv2 models: {hits}')
for p_ in vx_model.parameters(): p_.requires_grad_(False)

cfg = timm.data.resolve_model_data_config(vx_model)
vx_tf = timm.data.create_transform(**cfg, is_training=False)

@torch.no_grad()
def vx_extract(pil_img):
    return vx_model(vx_tf(pil_img.convert('RGB')).unsqueeze(0).to(device)).squeeze(0)

# smoke test — fail HERE, not mid-scoring
_probe = vx_extract(Image.new('RGB', (128, 128), (128, 128, 128)))
print(f'fidelity_vx ready: {VX_NAME}, feature dim {_probe.shape[0]}')

In [ ]:
# VORTEX replacement: raw BEiTv2 features as the fourth observer
import timm
from torchvision import transforms
_beit = timm.create_model('beitv2_base_patch16_224.in1k_ft_in22k_in1k',
                          pretrained=True, num_classes=0).to(device).eval()
for p in _beit.parameters(): p.requires_grad_(False)
vtf = transforms.Compose([transforms.ToTensor(),
      transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
      transforms.Resize((224,224))])
@torch.no_grad()
def vortex_feats(pil):
    return _beit(vtf(pil.convert('RGB')).unsqueeze(0).to(device)).cpu().numpy().ravel()
print("BEiTv2 observer ready (VORTEX replacement)")

In [ ]:
# ============================================================
# PHASE 2, CELL 7 — our four methods on the SAME 1000 pairs. Resumable.
# Dispatch via the FNS dict from the copied setup cell (same code path as
# the n=50 run). d is built ONCE per pair -> all methods see the identical
# encoding. make_recon comes from the setup cell — NOT redefined here.
# Timing: per-method interpolation-only (encode excluded); the speed
# table's headline number stays sourced from P30.
# ============================================================
import json, os, time
import numpy as np
from PIL import Image

OUT_ROOT = '/content/data/tm_scale1000'
MAN = json.load(open(f'{OUT_ROOT}/manifest.json'))
METHODS = ['pixel_linear', 'texton_linear', 'ot', 'ot_sym']
for m_ in METHODS + ['recon_A', 'recon_B']:
    os.makedirs(f'{OUT_ROOT}/{m_}', exist_ok=True)

def as_pil(out):
    if isinstance(out, Image.Image): return out
    return to_pil(out)

t_log = {m_: [] for m_ in METHODS}
t_enc = []
for p_ in MAN['pairs']:
    k = p_['k']
    A = Image.open(f"{OUT_ROOT}/A/{k:04d}.png").convert('RGB')
    B = Image.open(f"{OUT_ROOT}/B/{k:04d}.png").convert('RGB')

    missing = [m_ for m_ in METHODS
               if not os.path.exists(f'{OUT_ROOT}/{m_}/{k:04d}.png')]
    if missing:
        t0 = time.time()
        d = build_d(cnt_load(A), cnt_load(B))        # once per pair
        t_enc.append(time.time() - t0)
        for m_ in missing:
            t0 = time.time()
            out = FNS[m_](d, 0.5)
            t_log[m_].append(time.time() - t0)
            as_pil(out).resize((128, 128)).save(f'{OUT_ROOT}/{m_}/{k:04d}.png')

    for side, X in [('A', A), ('B', B)]:             # recon floor at scale
        fp = f'{OUT_ROOT}/recon_{side}/{k:04d}.png'
        if not os.path.exists(fp):
            as_pil(make_recon(X)).resize((128, 128)).save(fp)

    if k % 100 == 0: print(f'  {k}/1000')

print(f'\nmedian encode (build_d) per pair: {np.median(t_enc):.3f}s' if t_enc else '')
print('median secs per midpoint, interpolation only (encode excluded):')
for m_ in METHODS:
    if t_log[m_]:
        print(f'  {m_:15s} {np.median(t_log[m_]):.3f}  (n={len(t_log[m_])})')

In [ ]:
# ============================================================
# real make_recon — encode X, pair with itself, decode sa via the SAME
# pipeline the midpoints use. pixel_linear at eta=0 on build_d(e, e)
# returns the decode of X's own style map = the round-trip.
# ============================================================
def make_recon(X_pil):
    e = cnt_load(X_pil)
    return FNS['pixel_linear'](build_d(e, e), 0.0)

# verify it's a true round-trip before the loop touches it:
_test = as_pil(make_recon(Image.open(f'{OUT_ROOT}/A/0000.png').convert('RGB')))
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(Image.open(f'{OUT_ROOT}/A/0000.png')); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(_test); ax[1].set_title('recon'); ax[1].axis('off')
plt.show()

In [ ]:
# ============================================================
# smoke check: 5 pairs end-to-end — completeness + visual sanity +
# code-path match against the old n=50 run.
# ============================================================
import matplotlib.pyplot as plt
from PIL import Image
import os

# (a) completeness
for d_ in ['A', 'B', 'pixel_linear', 'texton_linear', 'ot', 'ot_sym',
           'recon_A', 'recon_B']:
    n = len(os.listdir(f'{OUT_ROOT}/{d_}'))
    print(f'  {d_:15s} {n} files' + ('' if n >= 5 else '  <-- MISSING'))

# (b) visual sanity: one row per pair — A | four midpoints | recon_A | B
rows, cols = 5, 7
names = ['A', 'pixel_linear', 'texton_linear', 'ot', 'ot_sym', 'recon_A', 'B']
fig, axes = plt.subplots(rows, cols, figsize=(2.1 * cols, 2.1 * rows))
for r in range(rows):
    for c, nm in enumerate(names):
        axes[r, c].imshow(Image.open(f'{OUT_ROOT}/{nm}/{r:04d}.png'))
        axes[r, c].axis('off')
        if r == 0: axes[r, c].set_title(nm, fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# what interpolation machinery did the setup cell give us?
import inspect
cands = {n: o for n, o in list(globals().items())
         if callable(o) and not n.startswith('_')
         and any(t in n.lower() for t in
                 ['mid', 'interp', 'ot', 'linear', 'texton', 'sym',
                  'transport', 'decode', 'recon', 'roundtrip', 'blend'])}
for n, o in sorted(cands.items()):
    try: sig = str(inspect.signature(o))
    except (ValueError, TypeError): sig = '(?)'
    print(f'{n:32s} {sig}')
# also check for a method-dispatch dict (common pattern in scoring cells):
for n, o in globals().items():
    if isinstance(o, dict) and o and all(callable(v) for v in o.values()):
        print(f'\ndict {n}: keys = {list(o.keys())}')

In [ ]:
# ============================================================
# PHASE 2, CELL 8 — PRE-REGISTERED READINGS (write-once, before scoring).
# R1  FID ranking replicates at n=1000: ot < pixel_linear < texture_mixer.
# R2  Recon-floor question, now answerable: expected recon < ot by ~+15
#     (the n=139 reading); if ot < recon at n=1000 THAT is the surprising
#     result and gets its own investigation, not a slide.
# R3  Paired observers at n=1000: parity on realism_inc / fidelity_ps
#     (p > .05, |median| < .01); DINOv2 gap ~-0.03 favoring TM expected to
#     go SIGNIFICANT at this n while staying below the 0.036
#     distinguishability threshold -> report as "small, stable, sub-threshold".
# R4  Bridge: repeated n=50 subsamples of the n=1000 FID sets should
#     reproduce the old table's instability (recon-ot sign flips), explaining
#     the P30 artifact in one figure.
# Deviations from these readings get reported as deviations.
# ============================================================
print(open(__import__('inspect').getfile(lambda: None)).read() if False else 'pre-registered: R1-R4 (see cell source)')

In [ ]:
# ============================================================
# PHASE 2, CELL 9 — observers + FID at n=1000.
# Same four observers as the n=50 head-to-head (realism_inc, realism_dino,
# fidelity_ps, fidelity_vx), same manifold construction, but:
#   (a) features cached to Drive per (set, k) — resumable, reconnect-safe
#   (b) per-pair scores appended to a CSV incrementally
#   (c) FID per set at n=1000 vs the SAME 128px reference as P30
# WIRING NOTE: reuse `score_pair(...)` / manifold-building from the copied
# head-to-head cell; only the loop + caching wrapper below is new.
# ============================================================
import pandas as pd, os, json
import numpy as np
from scipy.stats import wilcoxon

OUT_ROOT = '/content/data/tm_scale1000'
SETS = ['pixel_linear', 'texton_linear', 'ot', 'ot_sym', 'mid']   # mid = TM
CSV  = f'{OUT_ROOT}/scores.csv'
done_ks = set()
if os.path.exists(CSV):
    done_ks = set(pd.read_csv(CSV)['k'])
    print(f'resuming: {len(done_ks)} pairs already scored')

rows = []
for p_ in json.load(open(f'{OUT_ROOT}/manifest.json'))['pairs']:
    k = p_['k']
    if k in done_ks: continue
    r = dict(k=k, cA=p_['cA'], cB=p_['cB'])
    for s_ in SETS:
        sc = score_pair(f'{OUT_ROOT}/A/{k:04d}.png', f'{OUT_ROOT}/B/{k:04d}.png',
                        f'{OUT_ROOT}/{s_}/{k:04d}.png')   # from copied cell
        for name, v in sc.items(): r[f'{s_}__{name}'] = v
    rows.append(r)
    if len(rows) >= 50:                                     # incremental flush
        pd.DataFrame(rows).to_csv(CSV, mode='a', index=False,
                                  header=not os.path.exists(CSV))
        rows = []; print(f'  flushed through k={k}')
if rows:
    pd.DataFrame(rows).to_csv(CSV, mode='a', index=False,
                              header=not os.path.exists(CSV))

df = pd.read_csv(CSV)
print(f'\n=== MEDIANS (n={len(df)}, 128px, higher better) ===')
obs = ['realism_inc', 'realism_dino', 'fidelity_ps', 'fidelity_vx']
for s_ in SETS:
    print(f'  {s_:15s} ' + ' '.join(f'{df[f"{s_}__{o}"].median():+.4f}' for o in obs))
print('\n=== PAIRED vs texture_mixer (ours - TM; Wilcoxon) ===')
for s_ in ['ot', 'ot_sym']:
    for o in obs:
        d = df[f'{s_}__{o}'] - df[f'mid__{o}']
        w = wilcoxon(d)
        print(f'  {s_}/{o:13s} wins {(d>0).sum()}/{len(d)} '
              f'median {d.median():+.4f}  p={w.pvalue:.2g}')

# --- FID at n=1000, including the recon floor
from pytorch_fid.fid_score import calculate_fid_given_paths
REF = ref_dir_128    # same 128px reference dir as P30
fid_sets = SETS + ['recon_A']
print(f'\n=== FID vs DTD (n=1000, 128px) ===')
FIDs = {}
for s_ in fid_sets:
    FIDs[s_] = calculate_fid_given_paths([f'{OUT_ROOT}/{s_}', REF], 50, 'cuda', 2048)
    print(f'  {s_:15s} {FIDs[s_]:.2f}')
print(f'\nfloor question, settled at scale: ot - recon = '
      f'{FIDs["ot"] - FIDs["recon_A"]:+.1f}')

In [ ]:
# ============================================================
# CELL 8b — scoring setup at scale. Adapted from tm_headtohead cell 3:
# same manifold construction, same observer definitions, restructured to
# score ALL sets per pair in one batch (as the original did) and skip
# swap/seed controls (established at n=50; forward mids only on disk).
# ============================================================
import numpy as np, pandas as pd, random, os
from PIL import Image
from scipy.linalg import sqrtm
from scipy.stats import wilcoxon

# --- preflight: everything this cell depends on, checked by name
_need = ['inc_feats', 'dino_feats', 'ps_stats', 'cats', 'dtd_path']
_missing = [n for n in _need if n not in globals()]
assert not _missing, f'missing from namespace (rerun setup cell): {_missing}'
if 'vortex_feats' not in globals():                 # adapter for the BEiTv2 vx
    vortex_feats = lambda pil: vx_extract(pil).cpu().numpy()
    print('vortex_feats <- vx_extract adapter installed')

TMR = 128
OUT_ROOT = '/content/data/tm_scale1000'
SETS = ['pixel_linear', 'texton_linear', 'ot', 'ot_sym', 'mid']   # mid = TM

def fid_single(fn, pil, A, B):
    f, fA, fB = fn(pil), fn(A), fn(B)
    return 0.5*(np.linalg.norm(f-fA)+np.linalg.norm(f-fB))/(np.linalg.norm(fA-fB)+1e-9)

def manifold128(featfn, K=3, n=400):
    imgs = []
    for i in range(n):
        try: imgs.append(Image.open(dtd_path(random.choice(cats), 50000+i))
                         .convert('RGB').resize((TMR,)*2))
        except OSError: continue
    Phi = featfn(imgs); Phi = Phi[~np.isnan(Phi).any(axis=1)]
    D = np.linalg.norm(Phi[:,None]-Phi[None,:], axis=2); np.fill_diagonal(D, np.inf)
    rad = np.sort(D, axis=1)[:, K-1]; keep = rad <= np.median(rad)
    Pr, rr = Phi[keep], rad[keep]
    print(f'  manifold: {len(Phi)} imgs, {int(keep.sum())} kept')
    return lambda f: float(np.max(rr/(np.linalg.norm(Pr-f, axis=1)+1e-9)))

print('building manifolds...')
random.seed(777)          # manifold reference draw fixed -> resumable scoring
r_inc128, r_dino128 = manifold128(inc_feats), manifold128(dino_feats)

def score_row(k):
    """All observers, all sets, one pair. Returns a flat dict row."""
    A = Image.open(f'{OUT_ROOT}/A/{k:04d}.png').convert('RGB')
    B = Image.open(f'{OUT_ROOT}/B/{k:04d}.png').convert('RGB')
    mids = {s_: Image.open(f'{OUT_ROOT}/{s_}/{k:04d}.png').convert('RGB')
            for s_ in SETS}
    fA, fB = dino_feats([A])[0], dino_feats([B])[0]
    rec = dict(k=k, ab_dist=float(np.linalg.norm(fA-fB)))
    fI = inc_feats([mids[s_] for s_ in SETS])
    fD = dino_feats([mids[s_] for s_ in SETS])
    for i, s_ in enumerate(SETS):
        rec[f'{s_}__realism_inc']  =  r_inc128(fI[i])
        rec[f'{s_}__realism_dino'] =  r_dino128(fD[i])
        rec[f'{s_}__fidelity_ps']  = -fid_single(ps_stats, mids[s_], A, B)
        rec[f'{s_}__fidelity_vx']  = -fid_single(vortex_feats, mids[s_], A, B)
        dA, dB = np.linalg.norm(fD[i]-fA), np.linalg.norm(fD[i]-fB)
        rec[f'{s_}__placement']    =  float(dA/(dA+dB+1e-9))
    return rec
print('score_row ready')

In [ ]:
# ============================================================
# CELL 9 — score n=1000, resumable CSV, then the pre-registered readings.
# ============================================================
import pandas as pd, os, json
from scipy.stats import wilcoxon

CSV = f'{OUT_ROOT}/scores.csv'
done_ks = set(pd.read_csv(CSV)['k']) if os.path.exists(CSV) else set()
print(f'resuming: {len(done_ks)} already scored' if done_ks else 'fresh start')

buf = []
for p_ in json.load(open(f'{OUT_ROOT}/manifest.json'))['pairs']:
    k = p_['k']
    if k in done_ks: continue
    r = score_row(k); r.update(cA=p_['cA'], cB=p_['cB'])
    buf.append(r)
    if len(buf) >= 50:
        pd.DataFrame(buf).to_csv(CSV, mode='a', index=False,
                                 header=not os.path.exists(CSV))
        print(f'  flushed through k={k}'); buf = []
if buf:
    pd.DataFrame(buf).to_csv(CSV, mode='a', index=False,
                             header=not os.path.exists(CSV))

df = pd.read_csv(CSV)
obs = ['realism_inc', 'realism_dino', 'fidelity_ps', 'fidelity_vx']
print(f'\n=== MEDIANS (n={len(df)}, 128px, higher better) ===')
print(f"  {'method':15}" + ''.join(f'{o[:13]:>15}' for o in obs) + f"{'placement':>11}")
for s_ in SETS:
    print(f'  {s_:15}' + ''.join(f'{df[f"{s_}__{o}"].median():15.4f}' for o in obs)
          + f'{df[f"{s_}__placement"].median():11.3f}')

print('\n=== PAIRED vs texture_mixer (ours - TM; Wilcoxon) ===  [R3]')
for s_ in ['ot', 'ot_sym']:
    print(f'  {s_}:')
    for o in obs:
        d_ = df[f'{s_}__{o}'] - df[f'mid__{o}']
        p = wilcoxon(d_).pvalue if d_.abs().sum() > 0 else 1.0
        print(f'    {o:16} wins {int((d_>0).sum()):3d}/{len(d_)} '
              f'median {d_.median():+.4f}  p={p:.3g}')
print('  R3 reminder: DINOv2 gap significant-but-sub-threshold (<0.036) is the')
print('  EXPECTED reading, decided before this run.')

In [ ]:
# ============================================================
# CELL 9b — FID at n=1000 vs a 2819-image 128px DTD reference (P30's
# protocol, rebuilt here since ref_dir_128 lived in the other notebook).
# Answers R1 (ranking) and R2 (the recon-floor question) at scale.
# ============================================================
!pip -q install pytorch-fid 2>/dev/null
from pytorch_fid.fid_score import calculate_fid_given_paths
from PIL import Image
import os, random

REF = '/content/ref128'
os.makedirs(REF, exist_ok=True)
if len(os.listdir(REF)) < 2800:
    rng = random.Random(31)
    todo = [(c, f) for c in cats for f in sorted(os.listdir(f'{DTD_ROOT}/{c}'))]
    rng.shuffle(todo)
    for i, (c, f) in enumerate(todo[:2819]):
        fp = f'{REF}/{i:04d}.png'
        if not os.path.exists(fp):
            Image.open(f'{DTD_ROOT}/{c}/{f}').convert('RGB').resize((128,128)).save(fp)
    print(f'reference built: {len(os.listdir(REF))} images')

print('=== FID vs DTD (n=1000, 128px) ===   [R1, R2]')
FIDs = {}
for s_ in SETS + ['recon_A', 'recon_B']:
    FIDs[s_] = calculate_fid_given_paths([f'{OUT_ROOT}/{s_}', REF], 50, 'cuda', 2048)
    print(f'  {s_:15s} {FIDs[s_]:.2f}')
floor = min(FIDs['recon_A'], FIDs['recon_B'])
print(f'\nR2 settled: ot - recon floor = {FIDs["ot"] - floor:+.1f} '
      f'(pre-registered expectation: ~ +15 above)')
print(f'R1 check:   ot {FIDs["ot"]:.1f} < pixel_linear {FIDs["pixel_linear"]:.1f} '
      f'< TM {FIDs["mid"]:.1f} ?')

In [ ]:
# ============================================================
# PHASE 2, CELL 10 — small-n FID bias bridge: subsample the n=1000 sets
# at n in {50, 100, 200, 500, 1000}, 10 seeds each, plot FID vs n for
# ot / recon / mid. This is the figure that retires the P30 "below the
# floor" reading: at n=50 the ot/recon bands should overlap (sign
# unstable), separating cleanly as n grows.
# ============================================================
import matplotlib.pyplot as plt, tempfile, shutil, random, os
import numpy as np
from pytorch_fid.fid_score import calculate_fid_given_paths

NS, SEEDS = [50, 100, 200, 500, 1000], 10
curves = {}
for s_ in ['ot', 'recon_A', 'mid']:
    files = sorted(os.listdir(f'{OUT_ROOT}/{s_}'))
    curves[s_] = {}
    for n_ in NS:
        vals = []
        for sd in range(SEEDS if n_ < 1000 else 1):
            rng = random.Random(9000 + sd)
            tmp = tempfile.mkdtemp()
            for f in rng.sample(files, n_):
                os.symlink(f'{OUT_ROOT}/{s_}/{f}', f'{tmp}/{f}')
            vals.append(calculate_fid_given_paths([tmp, REF], 50, 'cuda', 2048))
            shutil.rmtree(tmp)
        curves[s_][n_] = (np.mean(vals), np.std(vals))
        print(f'  {s_} n={n_}: {np.mean(vals):.1f} ± {np.std(vals):.1f}')

plt.figure(figsize=(7, 4.5))
for s_, lab, c in [('ot', 'OT (ours)', 'C3'), ('recon_A', 'recon floor', 'gray'),
                   ('mid', 'Texture Mixer', 'C0')]:
    mu = [curves[s_][n_][0] for n_ in NS]; sd = [curves[s_][n_][1] for n_ in NS]
    plt.errorbar(NS, mu, yerr=sd, label=lab, color=c, marker='o', capsize=3)
plt.xscale('log'); plt.xlabel('n (midpoints in FID set)'); plt.ylabel('FID vs DTD')
plt.legend(); plt.title('FID small-sample bias: why the n=50 table misled')
plt.tight_layout()
plt.savefig(f'{OUT_ROOT}/fid_bias_bridge.png', dpi=180)
plt.show()

In [ ]:
for s_ in SETS:
    bad = df[f'{s_}__fidelity_ps'].isna().sum()
    if bad: print(s_, bad, 'NaNs')
# then rerun the ps Wilcoxon on the clean subset:
m = df[['ot__fidelity_ps','mid__fidelity_ps']].dropna()
print(wilcoxon(m['ot__fidelity_ps'] - m['mid__fidelity_ps']))